In [2]:
# ============================================================
# LADDER — Masked Annotation Semantic Similarity
# (AML terms removed before scoring)
# Panel A: Win counts  |  Panel B: Cosine similarity raincloud
# ============================================================
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggdist)
library(scales)

# === FILE PATH ===
csv_path  <- "Intermediate Files/AMLWITHOUTCONTEXTMASKED_Semantic_results_general_only.csv"
run_label <- "AML — Masked Annotation Analysis"

#keep_models <- c("BioLORD-2023", "MedCPT")
keep_models <- c("BioLORD-2023")
method_levels <- c("LADDER", "Hu et al", "GeneAgent")

method_pal <- c(
  "LADDER"   = "#a50026",
  "Hu et al" = "#0571b0",
  "GeneAgent" = "#4dac26"
)

theme_nature <- function(base_size = 10) {
  theme_classic(base_size = base_size, base_family = "Helvetica") +
  theme(
    panel.border       = element_rect(colour = "black", fill = NA, linewidth = 0.6),
    panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.35),
    panel.grid.minor   = element_blank(),
    axis.line          = element_blank(),
    axis.ticks         = element_line(colour = "black", linewidth = 0.45),
    axis.ticks.length  = unit(3, "pt"),
    axis.title         = element_text(face = "bold", size = base_size),
    axis.text          = element_text(colour = "black", size = base_size - 1),
    axis.text.x        = element_text(angle = 0, hjust = 0.5),
    legend.title       = element_text(face = "bold", size = base_size - 1),
    legend.text        = element_text(size = base_size - 2),
    legend.key         = element_blank(),
    legend.background  = element_blank(),
    plot.title         = element_text(face = "bold", size = base_size + 1, hjust = 0),
    plot.subtitle      = element_text(size = base_size - 2, colour = "grey45", hjust = 0),
    strip.background   = element_rect(fill = "grey96", colour = "black", linewidth = 0.5),
    strip.text         = element_text(face = "bold", size = base_size - 1),
    plot.margin        = margin(8, 12, 8, 8)
  )
}

# ============================================================
# LOAD & PREPARE
# ============================================================
raw <- read.csv(csv_path, stringsAsFactors = FALSE) %>%
  filter(Model %in% keep_models) %>%
  mutate(
    Model  = factor(Model, levels = keep_models),
    Winner = recode(Winner, "Our" = "LADDER", "Hu" = "Hu et al")
  )

# ============================================================
# PANEL A — Win counts 
# ============================================================
wins <- raw %>%
  count(Model, Winner) %>%
  rename(Method = Winner) %>%
  mutate(Method = factor(Method, levels = method_levels)) %>%
  complete(Model, Method, fill = list(n = 0))

pA <- ggplot(wins, aes(x = Model, y = n, fill = Method)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.65,
           colour = "white", linewidth = 0.4) +
  geom_text(aes(label = ifelse(n > 0, n, "")),
            position = position_dodge(width = 0.75),
            vjust = -0.4, size = 2.5, fontface = "bold", colour = "black") +
  scale_fill_manual(values = method_pal, drop = FALSE) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15)),
                     breaks = pretty_breaks(4)) +
  labs(
    title    = run_label,
    subtitle = "Embedding win counts (masked annotations)",
    x        = "Embedding Model",
    y        = "Number of Wins",
    fill     = "Method"
  ) +
  theme_nature() +
  theme(panel.grid.major.x = element_blank())

# ============================================================
# PANEL B — Cosine similarity raincloud
# ============================================================
sim_long <- raw %>%
  select(Model, Geneset,
         LADDER     = LADDER_Similarity,
         `Hu et al` = Hu_Similarity,
         GeneAgent  = GeneAgent_Similarity) %>%
  pivot_longer(c(LADDER, `Hu et al`, GeneAgent),
               names_to  = "Method",
               values_to = "Similarity") %>%
  mutate(Method = factor(Method, levels = method_levels),
         Model  = factor(Model,  levels = keep_models))

pB <- ggplot(sim_long, aes(x = Model, y = Similarity,
                            fill = Method, colour = Method)) +
  stat_halfeye(adjust = 0.8, width = 0.4, .width = 0,
               justification = -0.2, point_colour = NA, alpha = 0.72,
               position = position_dodge(width = 0.7)) +
  geom_boxplot(outlier.shape = NA, width = 0.16, linewidth = 0.45,
               colour = "black", alpha = 0.5,
               position = position_dodge(width = 0.7)) +
  stat_dots(side = "left", dotsize = 0.55, binwidth = 0.013,
            alpha = 0.3, position = position_dodge(width = 0.7)) +
  scale_fill_manual(values = method_pal, drop = FALSE) +
  scale_colour_manual(values = method_pal, drop = FALSE) +
  scale_y_continuous(breaks = seq(0, 1, 0.2), limits = c(NA, 1.05)) +
  labs(
    title    = run_label,
    subtitle = "Cosine similarity distribution (masked annotations)",
    x        = "Embedding Model",
    y        = "Cosine Similarity",
    fill     = "Method",
    colour   = "Method"
  ) +
  theme_nature() +
  guides(fill = "none", colour = "none")

# ============================================================
# ASSEMBLE & SAVE
# ============================================================
fig <- (pA | pB) +
  plot_layout(guides = "collect") &
  theme(legend.position  = "bottom",
        legend.direction = "horizontal")

ggsave("Fig_LADDER_AML_Masked.pdf", fig, width = 14, height = 5, dpi = 300)
ggsave("Fig_LADDER_AML_Masked.png", fig, width = 14, height = 5, dpi = 300)

cat("Saved Fig_LADDER_AML_Masked.pdf/.png\n")

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'AML — Masked Annotation Analysis' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'AML — Masked Annotation Analysis' in 'mbcsToSbcs': - substituted for — (U+2014)”


Saved Fig_LADDER_AML_Masked.pdf/.png
